# 04 · Visual RAG: charts and tables as ColPali image embeddings

Each chart, table or image page is embedded as an **image**: ColPali produces 1030 patch vectors × 128 dimensions. The vectors are stored in Qdrant as one multivector per page and searched with **exact MaxSim**. Here we use full vectors only, with no pooling; notebook 05 adds pooling.

**Tested:**
- chart questions from the eval set: NVIDIA FY2026 p. 60 (peer-comparison bar chart) and Chevron FY2022 p. 35 (production charts);
- descriptive visual queries;
- exact-number queries, where image search is expected to be weak.

**Optional:** `NB_VLM=1` lets a small vision model read the top page.

**Runtime:** **GPU required** for real ColPali (`vidore/colpali-v1.3-hf`, ~6 GB). Without a GPU the notebook uses a deterministic fake encoder, which tests the plumbing only and not retrieval quality.

In [ ]:
# 0. Setup. On Colab: clones the repo (branch below) and installs packages. Locally: run from inside the repo.
import json, os, subprocess, sys, time
from pathlib import Path

REPO_URL = "https://github.com/rama159065/Capstone.git"
BRANCH = "finance_report_qa_agent"
IN_COLAB = "google.colab" in sys.modules


def colab_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None


if IN_COLAB:
    ROOT = Path("/content/Capstone")
    if not ROOT.exists():
        token = colab_secret("GITHUB_TOKEN")          # needed while the repo is private
        url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url, str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "qdrant-client>=1.12,<2", "pymupdf>=1.24,<2",
                    "pydantic>=2", "pandas", "transformers>=4.57,<5", "accelerate"], check=True)
    for key in ("LLM_API_BASE", "LLM_API_KEY", "LLM_MODEL", "SEC_USER_AGENT"):
        if not os.getenv(key) and colab_secret(key):
            os.environ[key] = colab_secret(key)
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ.setdefault("SEC_USER_AGENT", "capstone-research your.name@example.com")   # SEC requires your real e-mail
print("repo:", ROOT, "| Colab:", IN_COLAB)

In [ ]:
import numpy as np
import pymupdf
from PIL import Image
from qdrant_client import QdrantClient, models
from scripts.common import notebook_helpers as nb
from scripts.rag import financial_ingest as fi

DOCS = ["NVDA_2026", "CVX_2022"]
WORK = Path(os.getenv("NB_WORK", "/content/nb04_visual" if IN_COLAB else "tmp/nb04_visual"))
MODE = os.getenv("NB_COLPALI", "auto")                 # auto | real | fake
MODE = ("real" if nb._gpu() else "fake") if MODE == "auto" else MODE
MAX_VISUAL_PAGES = int(os.getenv("NB_MAX_VISUAL_PAGES", "0")) or None
encoder = fi.ColPali(os.getenv("NB_COLPALI_MODEL", "vidore/colpali-v1.3-hf")) if MODE == "real" else nb.HashColPali()
print("ColPali:", MODE, "(fake = plumbing only, rankings are meaningless)" if MODE == "fake" else "")
rows = nb.pick_filings(DOCS)
pdfs = nb.fetch_pdfs(rows)

## Encode visual pages into a Qdrant multivector collection (full vectors, exact MaxSim)

In [ ]:
WORK.mkdir(parents=True, exist_ok=True)
client = QdrantClient(path=str(WORK / "qdrant"))
if client.collection_exists("visual_full"):
    client.delete_collection("visual_full")
client.create_collection("visual_full", vectors_config={"colpali": models.VectorParams(
    size=128, distance=models.Distance.COSINE,
    multivector_config=models.MultiVectorConfig(comparator=models.MultiVectorComparator.MAX_SIM))})


def render(document, number, scale=fi.RENDER_SCALE):
    pixmap = document[number - 1].get_pixmap(matrix=pymupdf.Matrix(scale, scale), alpha=False)
    return Image.frombytes("RGB", (pixmap.width, pixmap.height), pixmap.samples)


page_text, started, encoded = {}, time.time(), 0
for row in rows:
    pages = [p for p in fi.analyse_pdf(pdfs[row["document_id"]]) if p["visual"]][:MAX_VISUAL_PAGES]
    with pymupdf.open(pdfs[row["document_id"]]) as document:
        for start in range(0, len(pages), 4):
            batch = pages[start:start + 4]
            vectors = encoder.embed_images([render(document, p["page"]) for p in batch])
            client.upsert("visual_full", points=[models.PointStruct(
                id=fi.point_id(row["document_id"], p["page"], "nb04"), vector={"colpali": np.asarray(v, np.float32).tolist()},
                payload={"doc_id": row["document_id"], "page": p["page"], "page_type": p["page_type"]})
                for p, (v, _) in zip(batch, vectors)])
            for p in batch:
                page_text[(row["document_id"], p["page"])] = p["text"]
            encoded += len(batch)
print(f"{encoded} visual pages encoded in {time.time() - started:.0f}s ({encoded / (time.time() - started):.2f} pages/s)")

## Queries: is the right page on top?

In [ ]:
def search(query, k=10):
    started = time.perf_counter()
    points = client.query_points("visual_full", query=np.asarray(encoder.embed_query(query), np.float32).tolist(),
                                 using="colpali", limit=k).points
    return [(p.payload["doc_id"], p.payload["page"]) for p in points], time.perf_counter() - started


chart_items = nb.eval_items(DOCS, categories=("chart_value",), general=False)
nvda_revenue = nb.get_xbrl_fact("NVDA", "revenue", 2026)
QUERIES = [(i["question"], (s["doc_id"], s["page"]), "chart question") for i in chart_items
           for s in i["sources"] if s.get("page")][:6]
QUERIES += [("bar chart comparing revenue with peer companies", ("edgar_ars_NVDA_2026", 60), "descriptive"),
            ("oil and gas production by region chart", ("edgar_ars_CVX_2022", 35), "descriptive"),
            (f"revenue {nvda_revenue['value'] / 1e6:,.0f} million", None, "exact number")]
rows_out = []
for query, target, kind in QUERIES:
    ranked, seconds = search(query)
    rank = ranked.index(target) + 1 if target in ranked else None
    if target is None:                                  # exact-number query: does any top-3 page print the number?
        figure = f"{nvda_revenue['value'] / 1e6:,.0f}"
        rank = next((i + 1 for i, key in enumerate(ranked[:3]) if figure in page_text.get(key, "")), None)
    rows_out.append({"kind": kind, "query": query[:70], "target": target, "rank": rank, "top3": ranked[:3],
                     "ms": round(seconds * 1000)})
results = nb.table(rows_out)
nb.show(results)
print("hit@1:", (results["rank"] == 1).mean().round(2), "| hit@5:", results["rank"].le(5).mean().round(2))

In [ ]:
# Top page for the first chart question (thumbnail)
doc_id, page = search(QUERIES[0][0])[0][0]
with pymupdf.open(pdfs[doc_id]) as document:
    nb.show(render(document, page, scale=0.6))

## Optional: a small vision model reads the top page (`NB_VLM=1`, GPU)
This is an experiment only. Track B generation must use the provided model; in the design, a vision model's reading counts only when OCR and the geometry checks agree with it.

In [ ]:
if os.getenv("NB_VLM") == "1" and MODE == "real":
    from transformers import AutoModelForImageTextToText, AutoProcessor
    vlm_name = os.getenv("NB_VLM_MODEL", "Qwen/Qwen2-VL-2B-Instruct")
    processor = AutoProcessor.from_pretrained(vlm_name)
    vlm = AutoModelForImageTextToText.from_pretrained(vlm_name, device_map="cuda", dtype="auto")
    for item in chart_items[:3]:
        doc_id, page = search(item["question"])[0][0]
        with pymupdf.open(pdfs[doc_id]) as document:
            image = render(document, page)
        messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text":
                     item["question"] + " Answer with the numbers printed on the chart only; say 'not printed' if absent."}]}]
        inputs = processor(text=processor.apply_chat_template(messages, add_generation_prompt=True),
                           images=[image], return_tensors="pt").to("cuda")
        output = vlm.generate(**inputs, max_new_tokens=80)
        answer = processor.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        print(f"{item['id']} page {page}: {answer!r}\n   gold: {item['gold_answer'][:120]}")
else:
    print("skipped (set NB_VLM=1 on a GPU runtime)")

## Failure scenarios for image-embedding retrieval

| Failure | Why | How to see it |
|---|---|---|
| **Exact-number queries** | Patch embeddings encode how a page looks, not exact digits; "revenue 216,961" is weak evidence for an image | The `exact number` row is rarely at rank 1. Use text and sparse search, or XBRL, for figures |
| **Similar pages** | Many look-alike chart and table pages (segment tables, 5-year summaries) | `top3` contains siblings of the target page |
| **Long question wording** | Eval questions mention many entities; MaxSim rewards pages that share any of them | Chart questions rank lower than the short descriptive queries |
| **Cost** | 1030 × 128 vectors per page: ~0.26 MB at float16. Exact MaxSim scans every page | `ms` per query grows with the page count. Notebook 05 adds pooled prefetch for this |
| **The image is found, the value isn't** | Retrieval returns the page, not a number | The values must come from OCR (notebook 03), XBRL, or a validated vision reading |
| **No GPU** | The fake encoder gives random rankings | Check the `ColPali: fake` banner |